# CrossCodeEval adapter demo

Runs **one real CrossCodeEval example** through the full, unmodified pipeline:
loads the example (task_id/repository/file/prompt/groundtruth), clones and indexes
the referenced GitHub repo on demand, nominates candidates (BM25 + symbol +
dependency), lets Qwen pick useful ones, then StarCoder generates the completion.

Only `prompt` and `file` are ever passed to retrieval/selection/generation --
`groundtruth`/`right_context` are extracted but never fed downstream.

Uses the Hugging Face backends (not Ollama) -- proven more reliable on Colab.

**Before running**: Runtime -> Change runtime type -> select a GPU (T4 is fine).

In [ ]:
# Confirm a GPU is actually visible to this session -- if this errors or
# shows no GPU, Runtime -> Manage sessions -> terminate all, then reconnect.
!nvidia-smi

## 1. Get the project code

In [ ]:
%cd /content
import shutil, os
if os.path.exists("repo-code-completion"):
    shutil.rmtree("repo-code-completion")
!git clone "https://github.com/Robertkiza0/repo-code.git" repo-code-completion
%cd /content/repo-code-completion
!git log --oneline -3

## 2. Install dependencies

In [ ]:
!pip install -q tree-sitter tree-sitter-python tree-sitter-java tree-sitter-typescript tree-sitter-c-sharp rank-bm25 requests python-Levenshtein
!pip install -q torch transformers accelerate bitsandbytes

## 3. Hugging Face login

Add your token as a Colab Secret named `HF_TOKEN` (padlock icon, left sidebar)
before running this -- never pasted directly into the notebook.

In [ ]:
from huggingface_hub import login

try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception:
    login()  # prompts for the token interactively (input is hidden)

## 4. Load the example and its repository index

First run clones + indexes the referenced repo (cached under `data/cceval/` for
later runs). Looks up the example by `TASK_ID` rather than a positional index;
set it to any task_id from the 20-example sample (or pass a different
`jsonl_path` to draw from the full dataset -- requires the full CrossCodeEval
archive to be extracted, see the main README).

In [ ]:
from evaluation.cceval_adapter import DEFAULT_JSONL, find_example_index_by_task_id, load_cceval_example, locate_repo_index

TASK_ID = "project_cc_python/62"

EXAMPLE_INDEX = find_example_index_by_task_id(DEFAULT_JSONL, TASK_ID)
example = load_cceval_example(index=EXAMPLE_INDEX)
chunks = locate_repo_index(example["repository"])  # clones + indexes on first run, cached after
print("task_id:   ", example["task_id"])
print("repository:", example["repository"])
print("file:      ", example["file"])
print(f"{len(chunks)} chunks indexed")

## 5. Load the models (run this ONCE per session)

Loads Qwen (selection) and StarCoder (generation) onto the GPU. **Do not
re-run this cell** unless you've just restarted the kernel -- re-running it
without restarting loads a second copy of both models on top of whatever's
already resident, which alone can exhaust a T4's 15GB VRAM. If you need to
reload (e.g. to change `load_in_4bit` or the model), restart the kernel
first.

In [ ]:
from selection.backends import HuggingFaceBackend
from selection.llm_selector import LLMSelector
from generation.backends import HuggingFaceGenerationBackend
from generation.generator import CompletionGenerator

selector = LLMSelector(chunks, backend=HuggingFaceBackend())  # Qwen2.5-Coder-7B-Instruct, 4-bit
generator = CompletionGenerator(chunks, backend=HuggingFaceGenerationBackend())  # StarCoder2-3b, 4-bit
print("models loaded")

## 6. Verify isolation, run, and print (safe to re-run repeatedly)

Reuses the already-loaded `selector`/`generator` from section 5 -- this cell
does no model loading itself, so re-running it doesn't touch GPU memory
allocation the way section 5 does. Explicitly asserts/logs that groundtruth
was never used for retrieval/selection/generation (only for the final
comparison), and sanity-checks that the completion is genuinely generated
text rather than a leaked chunk source.

In [ ]:
import time
from evaluation.cceval_adapter import verify_and_run_task, print_experiment_log

t0 = time.time()
result = verify_and_run_task(TASK_ID, selector=selector, generator=generator)
print(f"took {time.time() - t0:.1f}s\n")

print_experiment_log(chunks, result)

## 7. Scale to 20 tasks (do not run a larger experiment yet)

Runs the same unmodified pipeline across all 20 tasks in the sample file
(reuses `selector`/`generator` from section 5 -- no reload). Only 4 unique
repos are referenced across all 20 tasks (already-cached `turboderp/exllama`
covers 14 of them), so this clones at most 3 new repos. A failing task is
recorded with its error and does not stop the run. Saves
`results/cceval_20_results.jsonl` and `results/cceval_20_summary.json`.

In [ ]:
from evaluation.experiment import preflight_check, print_summary_table, print_task_table, run_experiment

# Verify the 20 tasks + their repository/index mapping before running anything.
mapping = preflight_check(DEFAULT_JSONL, 20)
for m in mapping:
    print(f"  [{m['index']:2d}] {m['task_id']:<26} repo={m['repository']:<30} file={m['file']}")

In [ ]:
outcome = run_experiment(n_tasks=20, selector=selector, generator=generator)

print_summary_table(outcome["summary"])
print()
print_task_table(outcome["results"])

## 8. Inspect selection validity (before running another experiment)

Part A works on the just-saved `outcome["results"]` alone: verifies every
selected candidate label is actually a member of that task's own candidate
pool (making LLMSelector's built-in hallucination-filtering explicit and
checkable). Flags `selected_count == 0` cases -- a saved result alone can't
tell an intentional empty selection apart from a parsing failure.

Part B re-runs retrieval + selection ONLY (no generation, so it's fast) for
specific zero-selection task_ids, capturing Qwen's raw text response to
resolve that ambiguity definitively. Edit `ZERO_SELECTION_TASK_IDS` to match
whatever section 7 actually reported as `selected == 0` in your run.

In [ ]:
# Part A: structural validity check on the already-saved results.
from evaluation.experiment import check_selection_validity, print_selection_validity_table

validity_rows = check_selection_validity(outcome["results"])
print_selection_validity_table(validity_rows)

In [ ]:
# Part B: re-run selection only (fast, no generation) for zero-selection
# tasks to see Qwen's actual raw response and resolve the ambiguity.
from evaluation.experiment import inspect_task_selection, print_task_selection_diagnosis

ZERO_SELECTION_TASK_IDS = ["project_cc_python/67", "project_cc_python/75"]

for task_id in ZERO_SELECTION_TASK_IDS:
    diagnosis = inspect_task_selection(task_id, selector=selector)
    print_task_selection_diagnosis(diagnosis)
    print("=" * 70)